<a href="https://colab.research.google.com/github/adrinorosario/legal-pragmatic-inference/blob/main/hybrid_clause_and_term_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Clause and Vague Term Extraction

Working on this approach here, the goal is to extract the contractual clauses and the underlying vague terms in the particular clauses. For this, we will be used 3 datasets:

*   [CUAD](https://huggingface.co/datasets/theatticusproject/cuad)
*   [ContractNLI](https://huggingface.co/datasets/kiddothe2b/contract-nli); extensive information can be found [here](https://stanfordnlp.github.io/contract-nli/). Refer to the paper [here](https://arxiv.org/pdf/2110.01799)
*   [LEDGAR](https://huggingface.co/datasets/coastalcph/lex_glue) (the subset in LexGLUE)

Extracting the anchor clauses and the vague terms will form the first two parts of the [triplet dataset](https://github.com/adrinorosario/legal-pragmatic-inference/blob/main/docs/research/dataset_construction.md).




In [1]:
from IPython.display import HTML, display

def set_css():
    display(HTML('''
    <style>
        pre {
            white-space: pre-wrap;       /* CSS3 */
            white-space: -moz-pre-wrap;  /* Mozilla, since 1999 */
            white-space: -pre-wrap;      /* Opera 4-6 */
            white-space: -o-pre-wrap;    /* Opera 7 */
            word-wrap: break-word;       /* Internet Explorer 5.5+ */
        }
    </style>
    '''))

# Swapped 'pre_run' with 'pre_execute' to match modern IPython specifications
get_ipython().events.register('pre_execute', set_css)

In [30]:
import json
import requests
import re
import copy

import torch
from tqdm.auto import tqdm

## Extraction from CUAD

For this, we are using the JSON file from the HuggingFace page of the CUAD Dataset which can be found [here](https://huggingface.co/datasets/theatticusproject/cuad/tree/main/CUAD_v1)

In [3]:
# read the json file
cuad_v1_json = "/content/CUAD_v1.json"

with open(cuad_v1_json, "r") as file:
  data = json.load(file)

data.keys()

dict_keys(['version', 'data'])

In [4]:
len(data["data"]) # contains 510 entries

510

In [5]:
cuad_data = data["data"]
cuad_data[0].keys() # each entry, i.e., a contract contains the title and the paragraphs in it

dict_keys(['title', 'paragraphs'])

In [6]:
print(f"Type of cuad_data[0]['paragraphs']: {type(cuad_data[0]["paragraphs"])}")
print(f"Length of cuad_data[0]['paragraphs]: {len(cuad_data[0]["paragraphs"])}")

print(f"\nType of cuad_data[0]['paragraphs'][0]: {type(cuad_data[0]['paragraphs'][0])}")
print(f"Length of cuad_data[0]['paragraphs'][0]: {len(cuad_data[0]['paragraphs'][0])}")
print(f"Keys of cuad_data[0]['paragraphs'][0]: {cuad_data[0]["paragraphs"][0].keys()}")

Type of cuad_data[0]['paragraphs']: <class 'list'>
Length of cuad_data[0]['paragraphs]: 1

Type of cuad_data[0]['paragraphs'][0]: <class 'dict'>
Length of cuad_data[0]['paragraphs'][0]: 2
Keys of cuad_data[0]['paragraphs'][0]: dict_keys(['qas', 'context'])


1.   You are accessing each contract's paragraphs, which is a **list**.
2.   Each list of paragraphs contains a dictionary, which has two keys: **qas** and **context**



In [7]:
print(f"Type of cuad_data[0]['paragraphs'][0]['qas']: {type(cuad_data[0]['paragraphs'][0]['qas'])}")
print(f"Length of cuad_data[0]['paragraphs'][0]['qas']: {len((cuad_data[0]['paragraphs'][0]['qas']))}")

print("\nLooking at a single qas:")
print(cuad_data[0]['paragraphs'][0]['qas'][0])
print(f"Keys in a single qas: {cuad_data[0]['paragraphs'][0]['qas'][0].keys()}")

Type of cuad_data[0]['paragraphs'][0]['qas']: <class 'list'>
Length of cuad_data[0]['paragraphs'][0]['qas']: 41

Looking at a single qas:
{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}], 'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'is_impossible': False}
Keys in a single qas: dict_keys(['answers', 'id', 'question', 'is_impossible'])


In [8]:
cuad_data[0]['paragraphs'][0]['qas'][4]["answers"][0]

{'text': 'The term of this  Agreement  shall be ten (10)                            years (the "Term")  which shall  commence on the date                            upon which the Company  delivers to  Distributor  the                            last Sample, as defined  hereinafter.',
 'answer_start': 5268}

1.   **qas** is a list. Each item in the list is a dictionary.
2.   A single item in the **qas** list has the following keys: **answers**, **id**, **question**, and **is_impossible**.



In [9]:
cuad_data[0]['paragraphs'][0].keys()

dict_keys(['qas', 'context'])

In [10]:
# inspect the first contract's paragraph
cuad_data[0]["paragraphs"][0].keys() # each paragraph contains 'qas' and 'context'

# inspect the qas first; inspect the first item in the list
cuad_data[0]["paragraphs"][0]["qas"][0] # each qas item contains 'answers' which is a list of its own, 'id', 'question', and 'is_impossible'

# focussing on a single paragraph
single_paragraph = cuad_data[0]["paragraphs"][0]
# focussing on the same anchor text
anchor_text = single_paragraph["context"]

# loop through all questions inside the single paragraph
for qa in single_paragraph["qas"]:
  # skip categories where no terms were found
  if qa["is_impossible"]:
    continue

  clause_id = qa["id"].split("__")[-1]
  question = qa["question"]
  answers = [ans["text"] for ans in qa["answers"]]

  # print out the findings
  print(f"ID: {clause_id}")
  print(f"QUESTION: {question}")
  print(f"ANSWER: {answers}")
  print("="*40)

ID: Document Name
QUESTION: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
ANSWER: ['DISTRIBUTOR AGREEMENT']
ID: Parties
QUESTION: Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
ANSWER: ['Distributor', 'Electric City Corp.', 'Electric City of Illinois L.L.C.', 'Company', 'Electric City of Illinois LLC']
ID: Agreement Date
QUESTION: Highlight the parts (if any) of this contract related to "Agreement Date" that should be reviewed by a lawyer. Details: The date of the contract
ANSWER: ['7th day of September, 1999.']
ID: Effective Date
QUESTION: Highlight the parts (if any) of this contract related to "Effective Date" that should be reviewed by a lawyer. Details: The date when the contract is effective 
ANSWER: ['The term of this  Agreement  shall be ten (10)                        

In [11]:
anchor_text

'EXHIBIT 10.6\n\n                              DISTRIBUTOR AGREEMENT\n\n         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.\n\n                                    RECITALS\n\n         A. The  Company\'s  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.\n\n         B. Representations.  As an inducement to the Company to enter into this Agreement,  the  Distributor  has  represented  that  it has or  will  hav

In [12]:
# extract the unique clauses from the contracts
unique_clauses = set()

for contract in cuad_data:
  # access the paragraphs in each contract
  for paragraph in contract["paragraphs"]:
    # access the qas of each paragraph
    qas = paragraph["qas"]
    # extract the ids from each qa
    for qa in qas:
      if qa["is_impossible"]:
        continue
      clause_id = qa["id"].split("__")[-1]
      unique_clauses.add(clause_id)

print("Unique clauses found in the first 5 contracts:")
print(unique_clauses)
print(f"Number of unique clauses found: {len(unique_clauses)}")

Unique clauses found in the first 5 contracts:
{'Revenue/Profit Sharing', 'Affiliate License-Licensor', 'Liquidated Damages', 'Non-Disparagement', 'Joint Ip Ownership', 'Most Favored Nation', 'License Grant', 'Competitive Restriction Exception', 'Effective Date', 'Unlimited/All-You-Can-Eat-License', 'Volume Restriction', 'Minimum Commitment', 'Insurance', 'Rofr/Rofo/Rofn', 'Cap On Liability', 'Change Of Control', 'Governing Law', 'No-Solicit Of Employees', 'Uncapped Liability', 'Third Party Beneficiary', 'Exclusivity', 'Parties', 'Affiliate License-Licensee', 'Irrevocable Or Perpetual License', 'Source Code Escrow', 'Document Name', 'No-Solicit Of Customers', 'Termination For Convenience', 'Notice Period To Terminate Renewal', 'Non-Compete', 'Ip Ownership Assignment', 'Warranty Duration', 'Post-Termination Services', 'Agreement Date', 'Anti-Assignment', 'Audit Rights', 'Price Restrictions', 'Renewal Term', 'Expiration Date', 'Covenant Not To Sue', 'Non-Transferable License'}
Number of 

In [13]:
# HIGH PRIORITY CLAUSES THAT ARE SUBJECT TO INTENSE PRAGMATIC INFERENCE
# AND LITIGATION IN COURTS
tier_1_clauses = {
    'Audit Rights',
    'Termination For Convenience',
    'Most Favored Nation',
    'Non-Compete',
    'Insurance',
    'Minimum Commitment',
    'Post-Termination Services',
    'Warranty Duration'
}

# STRICT PROHIBITIONS BUT CAN ALSO BE LITIGATED IN COURTS BASED ON THE
# CONTEXT OF THE CONTRACT AND CASE
tier_2_clauses = {
    'Exclusivity',
    'Anti-Assignment',
    'Cap On Liability',
    'Competitive Restriction Exception',
    'Covenant Not To Sue',
    'No-Solicit Of Customers',
    'No-Solicit Of Employees',
    'Non-Disparagement',
    'Non-Transferable License',
    'Revenue/Profit Sharing',
    'Rofr/Rofo/Rofn',
    'Source Code Escrow',
    'Third Party Beneficiary',
    'Price Restrictions',
    'License Grant',
    'Affiliate License-Licensor',
    'Affiliate License-Licensee',
    'Ip Ownership Assignment',
    'Joint Ip Ownership',
    'Irrevocable Or Perpetual License',
    'Unlimited/All-You-Can-Eat-License',
    'Volume Restriction'
}

# DETERMINISTIC OPERATIONS AND CLAUSES; DATES, AMOUNT, NUMERICAL VALUES, AND
# OTHER CLEAR DETERMINISTIC OPERATIONS
tier_3_clauses = {
    'Agreement Date',
    'Document Name',
    'Effective Date',
    'Expiration Date',
    'Governing Law',
    'Liquidated Damages',
    'Notice Period To Terminate Renewal',
    'Renewal Term',
    'Parties',
    'Uncapped Liability'
}

In [14]:
vagueness_seed_set = {
    # ── Effort & diligence ──────────────────────────────────────────────
    "reasonable efforts", "best efforts", "commercially reasonable",
    "due diligence", "reasonable care", "good faith", "workmanlike manner",
    "best practices", "reasonable endeavours", "all reasonable steps",
    "every reasonable effort", "diligent efforts", "reasonable commercial efforts",
    "utmost care", "exercise of judgment", "commercially practicable",
    "economically reasonable", "technically feasible",
    "consistent with good industry practice", "as would a prudent operator",
    "acting reasonably", "using its discretion",

    # ── Time & urgency ──────────────────────────────────────────────────
    "promptly", "in a timely manner", "as soon as practicable",
    "without undue delay", "for a reasonable period",
    "termination of this Agreement", "from time to time", "periodic",
    "duration", "seasonable", "business hours", "within a reasonable time",
    "with all due speed", "expeditiously", "at the earliest opportunity",
    "without unnecessary delay", "within a commercially reasonable period",
    "in due course", "forthwith", "in due time", "on a timely basis",
    "in the near term", "shortly after", "when practicable",
    "upon reasonable notice", "reasonable notice period",

    # ── Scope, degree & quantity ────────────────────────────────────────
    "material", "substantial", "limited", "relevant", "related", "generally",
    "appropriate", "similar", "de minimis", "significant", "incidental",
    "including but not limited to", "inter alia", "and/or",
    "save as otherwise provided", "appreciable", "meaningful", "non-trivial",
    "measurable", "proportionate", "commensurate", "reasonably proportionate",
    "unduly burdensome", "reasonably necessary", "to the extent practicable",
    "to a reasonable extent", "without limitation", "as applicable",
    "where relevant", "as appropriate", "to the extent required",

    # ── Harm, change & threshold ────────────────────────────────────────
    "material adverse effect", "material breach", "material adverse change",
    "material adverse impact", "material adverse consequence",
    "materially and adversely", "substantial impairment", "material disruption",
    "material deviation", "materially prejudice", "disproportionate impact",
    "unreasonable hardship", "undue prejudice", "undue harm", "undue risk",

    # ── Necessity & discretion ──────────────────────────────────────────
    "necessary", "sole discretion", "need to know", "confidential nature",
    "adequate", "satisfactory", "proper", "intended purpose",
    "not to be unreasonably withheld", "mutual satisfaction", "at its option",
    "consultation", "absolute discretion", "unfettered discretion",
    "not to be unreasonably delayed", "not to be unreasonably conditioned",
    "without arbitrary restriction", "reasonably required",
    "reasonably requested", "if deemed appropriate", "as deemed necessary",
    "in its reasonable opinion", "acting in good faith",
    "in its reasonable judgment", "as it sees fit", "as directed",

    # ── Industry norms & quality ────────────────────────────────────────
    "customary", "ordinary course of business", "industry standard",
    "standard practice", "normally", "comparable", "acceptable",
    "conventional", "fit for purpose", "first-class condition",
    "commercially sensitive", "prevailing market practice",
    "generally accepted practice", "market standard",
    "accepted industry norms", "standard market terms",
    "customary market conditions", "in accordance with accepted methods",
    "consistent with past practice", "as is customary",
    "in accordance with best available techniques",
    "reasonable engineering standards", "professionally acceptable",
    "to a professional standard", "of merchantable quality",
    "of satisfactory quality",

    # ── Knowledge, intent & foresight ──────────────────────────────────
    "foreseeable", "contemplated", "intended", "anticipated", "applicable",
    "knowledge", "directly or indirectly", "disclosed in confidence",
    "all copies", "survive", "mutual agreement", "substantially similar",
    "reasonable expectations", "actual knowledge", "constructive knowledge",
    "reasonably should have known", "to the best of its knowledge",
    "as far as it is aware", "reasonably foreseeable",
    "unforeseen circumstances", "unanticipated events",
    "beyond reasonable expectation", "reasonable belief", "bona fide belief",
    "reasonable grounds", "having regard to all circumstances",

     # ── Confidentiality & information ───────────────────────────────────
    "proprietary information", "non-public information",
    "sensitive business information", "trade secrets",
    "sufficiently confidential", "reasonably considered confidential",
    "maintained in confidence", "treated as confidential",
    "in accordance with confidentiality obligations",

    # ── Financial & commercial terms ────────────────────────────────────
    "commercially attractive", "economically viable",
    "commercially justifiable", "at a reasonable price", "fair market value",
    "arm's length", "at prevailing rates", "on reasonable commercial terms",
    "on competitive terms", "at a rate reflecting market conditions",
    "at cost", "without unreasonable mark-up", "reasonable compensation",
    "reasonable fees",

    # ── Survival, agreement & modification ─────────────────────────────
    "notwithstanding the foregoing", "without prejudice to",
    "subject to the foregoing", "except as otherwise agreed",
    "unless otherwise specified", "where not inconsistent", "insofar as",
    "to the fullest extent permitted by law", "as may be amended",
    "as modified from time to time", "by mutual written consent",
}

len(vagueness_seed_set)

206

### Extracting clause categories and texts

Using the dataset available, the following needs to be extracted:

*   Document ID (for cross referencing later if needed)
*   Clause category/ID
*   Text from the clause that describes the contract
*   The tier it belongs to
*   The set of vague terms contained in it

Furthermore, we also need to check whether the clause is:

*   Lethal - belongs to tier 1
*   Has context risk - belongs to tier 2

All of these will be stored as a dictionary, and housed in a list



In [15]:
clause_and_terms = list()

for contract in cuad_data:
  # access the paragraphs in each contract
  for paragraph in contract["paragraphs"]:
    # access the qas of each paragraph
    qas = paragraph["qas"]
    # extract the ids from each qa
    for qa in qas:
      # check_lethality flags if the clause belongs to tier 1
      # context_risk flags if the clause belongs to tier 2
      check_lethality, context_risk = False, False
      tier = 0

      if qa["is_impossible"]:
        continue
      clause_id = qa["id"].split("__")[-1]

      # check if the clause id belongs to tier 3, if yes, discard it
      if clause_id in tier_3_clauses:
        continue

      # check tier and assign label
      if clause_id in tier_1_clauses:
        tier = 1
      elif clause_id in tier_2_clauses:
        tier = 2

      clause_text = qa["answers"][0]["text"]
      contract_id = contract['title']
      vague_terms = {term for term in vagueness_seed_set if term in clause_text}


      # check the lethality of the clause
      if tier == 1 and vague_terms:
        check_lethality = True
        context_risk = False
      elif tier == 2 and vague_terms:
        check_lethality = False
        context_risk = True

      if check_lethality or context_risk:
        clause_term_dict = {
            "contract_id": contract_id,
            "clause_category": clause_id,
            "clause_text": clause_text,
            "tier": tier,
            "vague_terms": vague_terms,
            "is_lethal": check_lethality,
            "has_context_risk": context_risk
        }

        clause_and_terms.append(clause_term_dict)

In [16]:
print(f"Extracted data: {len(clause_and_terms)}")

Extracted data: 1428


In [17]:
from prompt_toolkit.shortcuts import print_container
import random

sampling_size = int(len(clause_and_terms) * 0.20)
compressed_sampling_size = int(sampling_size * 0.05)

for i in range(compressed_sampling_size):
  random_idx = random.randint(0, len(clause_and_terms)-1)

  print(f"CLAUSE CATEGORY: {clause_and_terms[random_idx]["clause_category"]}")
  print(f"CLAUSE TEXT: {clause_and_terms[random_idx]["clause_text"]}")
  print(f"TIER: {clause_and_terms[random_idx]["tier"]}")
  print(f"VAGUE TERMS: {clause_and_terms[random_idx]["vague_terms"]}")
  print(f"IS LETHAL: {clause_and_terms[random_idx]["is_lethal"]}")
  print(f"HAS CONTEXT RISK: {clause_and_terms[random_idx]["has_context_risk"]}")
  print("="*40, end="\n\n")

CLAUSE CATEGORY: Post-Termination Services
CLAUSE TEXT: Turpin shall  deliver to the Company all Work      Product and Company  Property,  including all originals and copies thereof,      in Turpin's possession and/or control,  at the request of the Company,  or,      in the  absence  of  such a  request,  upon  the  termination  of  Turpin's      employment with the Company.
TIER: 1
VAGUE TERMS: {'and/or'}
IS LETHAL: True
HAS CONTEXT RISK: False

CLAUSE CATEGORY: License Grant
CLAUSE TEXT: SONY, on behalf of itself and its Affiliates, hereby grants to PURCHASER a worldwide, non-exclusive, fully paid-up, royalty-free  license (a) under the Licensed Patents to make, have made, use, offer to sell, sell, otherwise dispose of, and import any Competing Products  (including, without limitation, the SRAM Products); and (b) to use, reproduce, modify, prepare derivative works of, perform, display, and otherwise  practice and exploit in any manner any and all of the SRAM Intellectual Property in

Right now, each data point `clause_and_terms` houses the vague clauses, vague terms, and the accompanying signals such as lethality and context risks.

From this, we move on to semantic mapping with the case law from the Harvard Corpus, i.e., [COLD Cases](https://huggingface.co/datasets/harvard-lil/cold-cases).

The `clause_text` present in each data point will be used to semantically map the right case law from this corpus which will be the judicial prose of the triplet.



## Semantic Mapping with Harvard LIL COLD Cases

Taking `clause_and_terms`, we will use the `clause_text` key/item and encode it into its **vector embeddings**.

A judge will not always write the contractual clauses as and how it appears in a contract in his reasoning; hence, string matching will fail. We need to map them in a dense **vector space**.

In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np

# use the latest embedding models available in huggingface
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# vectorize the clause_texts in each data point in clause_and_terms
anchor_clause_texts = [data_point["clause_text"] for data_point in clause_and_terms]
anchor_clause_embeddings = embedding_model.encode(
    anchor_clause_texts,
    show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

## Dual-Vector Anchoring

### Implementing a coarse filter on harvard-lil/cold-cases

The coarse filter is not looking for contratual obligations here. Rather, it's focus is to filter out all sentences from the opinion texts that are not useful for the vector embeddings to compute similarity with.

From a micro perspective, the filter will:

*   Check if the opinion texts belong to **contract, commercial law, corporate law, or other such similar cases.** No statutory, administrative, or tort law cases will be considered.
*   Segment each opinion text into individual sentences. Use sentences that have $> 10$ and $\le 100$ words in a sentence. This can be tuned depending on the quality of the filter's output.
*   Ensure **no statutory sentences** are present. *This is important.*
*   No checking for modals or seed words (vague terms) in this step.





In [19]:
# Statutory signals
STATUTORY_PATTERN = re.compile(
    r'\b(§|Code|Section|Article|Constitution|Statute|Act of \d{4}|'
    r'U\.S\.C\.|W\.Va\. Code|provided by law|mandates that|'
    r'every municipality|any person|no employee|all employers)\b',
    re.IGNORECASE
)

In [20]:
def filter_cold_cases_opinion_text(opinion_text: str):
  """
  """

  # store all valid sentences individually, not as an entire opinion text
  clause_sentences = []
  reasoning_sentences = []

  # structural indicators for retrospective reasoning prose
  REASONING_INDICATORS = re.compile(
    # Epistemic / judgment verbs (strong signals)
    r'\b(held|concluded|found|determined|argued|asserted|interpreted|appealed|testified)\b|'
    r'\b(finds|found|ruled|decided|opined|considered|recognised|acknowledged)\b|'
    r'\b(construed|construction|construing)\b|'

    # First-person judicial voice (strong signals)
    r'\b(I\s+find|I\s+am\s+satisfied|I\s+consider|I\s+prefer|I\s+accept|I\s+reject)\b|'
    r'\b(I\s+agree|I\s+disagree|I\s+conclude|I\s+hold|in\s+my\s+judgment|in\s+my\s+view)\b|'
    r'\b(the\s+court\s+finds|the\s+court\s+held|the\s+court\s+concludes|the\s+court\s+considers)\b|'

    # Plain meaning / interpretive analysis (strong signals)
    r'\b(on\s+a\s+proper\s+construction|the\s+plain\s+meaning|the\s+natural\s+meaning)\b|'
    r'\b(purposive|contextual\s+reading|read\s+as\s+a\s+whole|read\s+together)\b|'
    r'\b(the\s+better\s+view|the\s+correct\s+interpretation|properly\s+construed)\b|'

    # Inferential / consequential connectors (medium signals)
    r'\b(therefore|accordingly|it\s+follows|consequently|necessarily\s+means)\b|'
    r'\b(it\s+is\s+clear\s+that|it\s+must\s+follow|this\s+suggests|this\s+indicates)\b|'
    r'\b(would\s+have|could\s+have|should\s+have|must\s+have)\b|'

    # Reasonableness / obligation framing (medium signals)
    r'\b(reasonable|reasonably|unreasonable|unreasonably)\b|'
    r'\b(the\s+parties\s+intended|the\s+intention\s+of\s+the\s+parties|objectively\s+construed)\b|'
    r'\b(implied\s+term|implied\s+obligation|necessary\s+implication)\b|'

    # Comparative / distinguishing reasoning (medium signals)
    r'\b(unlike|by\s+contrast|whereas|distinguished\s+from|analogous\s+to)\b|'
    r'\b(consistent\s+with|inconsistent\s+with|contrary\s+to)\b|'

    # Original weak signals retained
    r'\b(this\s+provision|such\s+obligation|the\s+disputed|underlying\s+contract)\b',

    re.IGNORECASE
  )

  # split the entire opinion text into individual sentences
  sentences = re.split(r'(?<=[.!?])\s+', opinion_text)

  for sentence in sentences:
    sentence = sentence.strip() # strip off all leading and trailing whitespaces

    # check for statutory patterns and skip them
    if STATUTORY_PATTERN.search(sentence):
      continue

    # check the number of words in the sentence and eliminate accordingly
    words = sentence.split()
    if len(words) < 10 or len(words) > 100:
      continue

    # the gatekeeper routing to embedding or RRL
    if REASONING_INDICATORS.search(sentence):
      reasoning_sentences.append(sentence)
    else:
      clause_sentences.append(sentence)

  # check if valid sentences is not empty, and batch encode it
  if clause_sentences:
    valid_sentences_embeddings = embedding_model.encode(
        clause_sentences,
        show_progress_bar=False,
        batch_size=64
    )

    # check the cosine similarity with the cuad anchors
    similarities = embedding_model.similarity(
        anchor_clause_embeddings,
        valid_sentences_embeddings
    )

    output = {
        "opinion_text": opinion_text,
        "sentences": clause_sentences,
        "similarities": similarities,
        "reasoning_sentences": reasoning_sentences
    }

    return output

  else:
    return False

In [21]:
from datasets import load_dataset

# load the dataset from HuggingFace: https://huggingface.co/datasets/harvard-lil/cold-cases
cold_cases = load_dataset(
    "harvard-lil/cold-cases",
    split="train",
    streaming=True
)

cold_cases["opinions"]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

In [22]:
# parse the dataset and check for all the different case natures
case_natures = set()

# get all the different natures of cases
for idx, case in enumerate(cold_cases):
  if idx >= 50000:
    break
  nature = case.get("nature_of_suit", "") or "" # read the nature of the case
  case_natures.add(nature)

len(case_natures), case_natures

(52,
 {'',
  '1581(a)',
  '1581(c)',
  'Administrative Agency-Other',
  'Agency',
  'Bankruptcy',
  'Bankruptcy Direct from BC',
  'CER',
  'CON',
  'Civil',
  'Civil Rights',
  'Civil-Other',
  'Colorado Consumer Protection Act—Deceptive Trade Practice—Disclosure',
  'Criminal',
  'Criminal-Other',
  'DIVORCE/PROPERTY DIV./ALIMONY',
  'DRUGS/CONTRABAND',
  'Direct Appeal',
  'Direct Criminal',
  'Felony (non-Death Penalty)',
  'Frap 9(a)',
  'HOMICIDE',
  'Habeas',
  'Immigration',
  'Juvenile',
  'MALPRACTICE',
  'Magistrate Case',
  'NEW',
  'Non Direct Criminal',
  'OIL, GAS AND MINERALS',
  'ORD',
  'OTHER (Civil)',
  'Original Proceeding',
  'POST-CONVICTION RELIEF',
  'Post-Conviction Appeal',
  'Prisoner',
  'Prisoner w/ Counsel',
  'Prisoner w/ out Counsel',
  'Private Civil Diversity',
  'Private Civil Federal',
  'Professional Regulation',
  'Tax Court',
  'Tort, Contract, and Real Property',
  'United States Civil',
  'Workers Compensation',
  'Writ Application-Other',
  'a

In [23]:
target_case_natures = {
    'Tort, Contract, and Real Property',
    'Private Civil Diversity',
    'Private Civil Federal',
    'OIL, GAS AND MINERALS',
}

In [24]:
count = 0

for case in cold_cases:
  if count >= 10:
    break

  nature = case.get("nature_of_suit", "") or "" # read the nature of the case
  # only proceed with the targetted case natures
  if nature in target_case_natures:

      opinions = case.get("opinions", []) # extract the opinions into a list

      # check if the case has an opinion text that can be extracted
      if opinions and opinions[0].get("opinion_text"):
        validity_status = filter_cold_cases_opinion_text(opinions[0]['opinion_text'])

        # if the filter returns false, skip document
        if not validity_status:
          continue

        # here now, we need to isolate the high scoring coordinates
        # if the cosine similarity >= 70
        # pair the raw case sentence text directly with the metadata of the
        # matching CUAD anchor clause
        row_indices, col_indices = torch.where(validity_status['similarities'] >= 0.70)

        for row, col in zip(row_indices, col_indices):
          row_idx = row.item()
          col_idx = col.item()

          # extract the matching score from the matrix
          score = validity_status['similarities'][row_idx][col_idx].item()

          # obtain the matching cuad metadata from the json directly
          # row_idx looks up the cuad anchors
          matching_cuad_metadata = clause_and_terms[row_idx]

          # col_idx looks up the raw sentence from the valid sentences
          matching_raw_sentence = validity_status["sentences"][col_idx]

          print(f"Matching Score: {score: .2f}")
          print(f"|-- CUAD CATEGORY: {matching_cuad_metadata["clause_category"]}")
          print(f"|-- CUAD Anchor: {matching_cuad_metadata["clause_text"]}")
          print(f"└── Raw sentence: {matching_raw_sentence}")
          print("="*50)

          count += 1

Matching Score:  0.71
|-- CUAD CATEGORY: Termination For Convenience
|-- CUAD Anchor: Notwithstanding the foregoing, this Agreement shall be earlier terminated (x) by mutual agreement of the parties, or (y) at any time upon sixty (60) days advance written notice to the other party.
└── Raw sentence: Rather, the Advisory Agreements provided for termination without cause upon
sixty days’ written notice.
Matching Score:  0.73
|-- CUAD CATEGORY: Cap On Liability
|-- CUAD Anchor: In no event shall either party be liable to the other for the recovery of any special, indirect or consequential damages even if the defendant party had been advised of the possibility of such damages including but not limited to lost profits, lost revenues, failure to realize expected savings, loss of data and loss of use.
└── Raw sentence: If no defense applies, “the responsible party will always bear first-
level liability, but will be able to recover over against third parties either
through contribution accord

Check the nature of the cosine similarity of outputs for following thresholds:

*   $> 60\% \text{ and } \le 70\%$
*   $> 70\% \text{ and } \le 75\%$
*   $> 75\%$





**Notes until this point in the extraction:**

1.  The threshold bucket that contains scores higher than 75% contains pure lexical matchings which are similar to the substring matching that can be achieved using pure code
2.  The bucket with scorings in the 70-75% range house a lot of text is judicial reasoning, rather than the clauses.
    *   The sentences assertive, stating the judge's interpretation of the clauses and its obligations, **not** the clause itself
    *   These are mappings that are needed for the interpretation part of the triplet structure which will what the reward system will rely on
    *   Failure to isolate these two aspects will corrupt the learning process of the reinforcement learning loop
3.  Surprisingly, the 60-70% bucket contains positive signals where it had identified clauses, some of then including:

    ```
    |-- CUAD CATEGORY: Termination For Convenience
    |-- CUAD Anchor: Either party may, at its option, terminate this Agreement       without cause, effective at any time after January 31, 1999, upon giving       at least ninety (90) days prior written notice of such termination to the       other party.
    └── Raw sentence: Rather, the Advisory Agreements provided for termination without cause upon sixty days’ written notice.
    ```

    This shows promising results. It also contains a lot of structural noise compared to the other two buckets.

In [25]:
distributed_threshold_results = {
    0.60: [],
    0.70: [],
    "reasoning_sentences": []
}

# target_count_per_bucket = 50

# Initialize tqdm for the count of valid cases
valid_cases_pbar = tqdm(total=100, desc="Valid Cases Processed", unit="case")

for case in tqdm(cold_cases, desc="Processing cold cases"): # Wrap cold_cases with tqdm
  # # Check if all clause-related buckets are full, then break the outer loop
  # if all(len(distributed_threshold_results[key]) >= target_count_per_bucket for key in [0.60, 0.70]):
  #   break
  if valid_cases_pbar.n >= 100:
    break

  nature = case.get("nature_of_suit", "") or "" # read the nature of the case
  # only proceed with the targetted case natures
  if nature in target_case_natures:

      opinions = case.get("opinions", [])
      # check if the case has an opinion text that can be extracted
      if opinions and opinions[0].get("opinion_text"):
        validity_status = filter_cold_cases_opinion_text(opinions[0]['opinion_text'])

        # if the filter returns false, skip document
        if not validity_status:
          continue

        # update the count value and the progress bar
        valid_cases_pbar.update(1)

        # Collect reasoning sentences if they exist
        if validity_status["reasoning_sentences"]:
            distributed_threshold_results["reasoning_sentences"].extend(validity_status["reasoning_sentences"])

        # Proceed with clause sentences if any were returned (similarities are calculated only if sentences are present)
        if validity_status["sentences"]:
            # Always check for similarities >= 0.60 and then distribute
            row_indices, col_indices = torch.where(validity_status['similarities'] >= 0.60)

            for row, col in zip(row_indices, col_indices):
              row_idx = row.item()
              col_idx = col.item()

              score = validity_status['similarities'][row_idx][col_idx].item()

              matching_cuad_metadata = clause_and_terms[row_idx]

              matching_raw_sentence = validity_status["sentences"][col_idx]

              matched_sentence_anchor = {
                  "cuad_category": matching_cuad_metadata["clause_category"],
                  "cuad_anchor": matching_cuad_metadata["clause_text"],
                  "raw_sentence": matching_raw_sentence
              }

              # Distribute based on score, only if the respective bucket is not full
              if score > 0.60 and score <= 0.70:
                # if len(distributed_threshold_results[0.60]) < target_count_per_bucket:
                distributed_threshold_results[0.60].append(matched_sentence_anchor)
              elif score > 0.70:
                # if len(distributed_threshold_results[0.70]) < target_count_per_bucket:
                distributed_threshold_results[0.70].append(matched_sentence_anchor)

valid_cases_pbar.close() # Close the progress bar when done
print(f"Number of objects with > 60% similarity: {len(distributed_threshold_results[0.60])}")
print(f"Number of objects with > 70% similarity: {len(distributed_threshold_results[0.70])}")
print(f"Number of reasoning sentences collected: {len(distributed_threshold_results['reasoning_sentences'])}")

Valid Cases Processed:   0%|          | 0/100 [00:00<?, ?case/s]

Processing cold cases: 0it [00:00, ?it/s]

Number of objects with > 60% similarity: 264
Number of objects with > 70% similarity: 11
Number of reasoning sentences collected: 2121


In [26]:
rand_idx = random.randint(0, len(distributed_threshold_results[0.60]))

for i in range(7):
    rand_idx = random.randint(0, len(distributed_threshold_results[0.60]))
    data_point = distributed_threshold_results[0.60][rand_idx]
    print(f"|-- CUAD CATEGORY: {data_point["cuad_category"]}")
    print(f"|-- CUAD Anchor: {data_point["cuad_anchor"]}")
    print(f"└── Raw sentence: {data_point["raw_sentence"]}")
    print("="*50, end="\n\n")

|-- CUAD CATEGORY: Cap On Liability
|-- CUAD Anchor: Neither Party shall be liable to the other for any special, indirect, incidental, consequential, punitive or exemplary damages, including, but not limited to, lost profits, even if such Party alleged to be liable has knowledge of the possibility of such damages, provided, however, that the limitations set forth in this Section shall not apply to or in any way limit the obligations of the Section entitled "Indemnity," the Section entitled "Confidentiality and Information Protection," or Supplier's gross negligence or willful misconduct.
└── Raw sentence: They provide an insured
with indemnification for damages up to policy limits for which the insured becomes liable as a
result of tort liability to a third party.” Chester-O’Donley & Assoc., 972 S.W.2d at 6 (citations
omitted).

|-- CUAD CATEGORY: Cap On Liability
|-- CUAD Anchor: Except for claims arising under section 6, in no event will either party be liable for any special, indire

In [27]:
rand_idx = random.randint(0, len(distributed_threshold_results[0.70]))

for i in range(7):
    rand_idx = random.randint(0, len(distributed_threshold_results[0.70]))
    data_point = distributed_threshold_results[0.70][rand_idx]
    print(f"|-- CUAD CATEGORY: {data_point["cuad_category"]}")
    print(f"|-- CUAD Anchor: {data_point["cuad_anchor"]}")
    print(f"└── Raw sentence: {data_point["raw_sentence"]}")
    print("="*50, end="\n\n")

|-- CUAD CATEGORY: Cap On Liability
|-- CUAD Anchor: Neither Party shall be liable to the other for any special, indirect, incidental, consequential, punitive or exemplary damages, including, but not limited to, lost profits, even if such Party alleged to be liable has knowledge of the possibility of such damages, provided, however, that the limitations set forth in this Section shall not apply to or in any way limit the obligations of the Section entitled "Indemnity," the Section entitled "Confidentiality and Information Protection," or Supplier's gross negligence or willful misconduct.
└── Raw sentence: If no defense applies, “the responsible party will always bear first-
level liability, but will be able to recover over against third parties either
through contribution according to principles of comparative fault or by
invoking a hold harmless or indemnification agreement, if applicable.” See 2
THOMAS J.

|-- CUAD CATEGORY: Termination For Convenience
|-- CUAD Anchor: Notwithstandin

In [28]:
for i in range(7):
    rand_idx = random.randint(0, len(distributed_threshold_results["reasoning_sentences"]))
    data_point = distributed_threshold_results["reasoning_sentences"][rand_idx]
    print(data_point)
    print("="*80)

Based largely on those findings regarding the conduct and motivations
of the TPSA, the district court concluded that the Texas legislature enacted the
public corporation ban with the same protectionist motivations.
1983) (holding that the appealed order must “finally
determine the rights or liabilities of either party to this dispute”); MS Tabea
Schiffahrtsgesellschaft MBH & Co.
The district court also determined that the public corporation ban does
not violate the Equal Protection Clause because the ban is rationally related
to the state’s legitimate purpose of reducing the availability and consumption
of liquor throughout Texas.
The district court found that although the Companies did
not make an “exhaustive effort,” they made a good-faith effort to confer with
the County.
Levine’s counsel, stating “I want to confirm that this settlement is to include

the Rockwool [Respondents] consistent with dismissal of all claims set forth in 19-C-139.
We, therefore,
arrive at the identical conc

Right now, we have a lot more judicial reasoning sentences that try to interpret the clauses than the actual clauses themselves. If we were to create a dataset using this, we would end up with a highly **imbalanced dataset.** However, there is something else that we can do, one that can highly speed up this process of constructing the triplet structure.

## Reverse Constructing the JDAR Triplet

Given that we already have the anchor clauses, i.e., the anchor clause, clause text, and the vague terms that were extracted from CUAD. Using these clause texts and the judical reasoning candidates that we have, we can use vector spaces to pair the clause texts with their judicial interpretation counterparts.

In essence, the triplet has the following structure:

$
\text{JDAR Triplet Data Point} = \begin{cases}
\textbf{Clause} : & \text{Clean CUAD Text Snippet} \\
\textbf{Terms} : & \text{Extracted Seed Words from the CUAD snippet and the judicial interpretation} \\
\textbf{Reasoning} : & \text{Aligned Harvard-LIL COLD Cases Opinion Sentence (i.e., the extracted reasoning sentences}
\end{cases}
$

In [29]:
cold_cases_reasoning_sentences = distributed_threshold_results["reasoning_sentences"]

# encode/vectorize the reasoning sentences
reasoning_sentence_embeddings = embedding_model.encode(
    cold_cases_reasoning_sentences,
    show_progress_bar=True
)

len(reasoning_sentence_embeddings) == len(cold_cases_reasoning_sentences)

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

True

Compute the similarities between **reasoning_sentence_embeddings** and **anchor_clause_embeddings**. From the matrix computed, choose those with the highest score, say $\ge 0.72$, and pair it with the corresponding anchor clause in **clause_and_terms.**

In [33]:
# compute similarity
reasoning_similarities = embedding_model.similarity(
    anchor_clause_embeddings,
    reasoning_sentence_embeddings
)

sample_triplet_set = list()
# Lowered threshold to 0.65 based on max similarity of 0.6866 as found using the cell below
threshold = 0.65

# Filter indices first to provide a total for the progress bar
row_idx, col_idx = torch.where(reasoning_similarities >= threshold)
matches = list(zip(row_idx, col_idx))

print(f"Found {len(matches)} potential pairs with similarity >= {threshold}")

# Progress bar for the construction process
for row, col in tqdm(matches, desc="Constructing Triplets", unit="triplet"):
  row_index = row.item()
  col_index = col.item()

  # use deepcopy so we don't modify the original 'clause_and_terms' list
  matching_cuad_metadata = copy.deepcopy(clause_and_terms[row_index])

  # extract the matching reasoning sentence
  matching_reasoning_sentence = cold_cases_reasoning_sentences[col_index]

  clause_text = matching_cuad_metadata["clause_text"]

  # extract the vague terms from both the cuad clause text and the reasoning sentences
  terms_in_reasoning = [term for term in vagueness_seed_set if term in matching_reasoning_sentence]

  # append the reasoning sentence and replace the vague terms with the newly
  # extracted terms from the reasoning
  matching_cuad_metadata["reasoning_sentence"] = matching_reasoning_sentence
  matching_cuad_metadata["reasoning_vague_terms"] = terms_in_reasoning

  sample_triplet_set.append(matching_cuad_metadata)

print(f"Successfully constructed {len(sample_triplet_set)} triplet data points.")

Found 12 potential pairs with similarity >= 0.65


Constructing Triplets:   0%|          | 0/12 [00:00<?, ?triplet/s]

Successfully constructed 12 triplet data points.


In [34]:
# inspect the actual distribution of similarity scores
max_score = reasoning_similarities.max().item()
mean_score = reasoning_similarities.mean().item()

print(f"Maximum similarity found: {max_score:.4f}")
print(f"Average similarity: {mean_score:.4f}")

for test_threshold in [0.65, 0.68, 0.70]:
    count = (reasoning_similarities >= test_threshold).sum().item()
    print(f"Pairs with similarity >= {test_threshold}: {count}")

Maximum similarity found: 0.6866
Average similarity: 0.1434
Pairs with similarity >= 0.65: 12
Pairs with similarity >= 0.68: 1
Pairs with similarity >= 0.7: 0


In [35]:
sample_triplet_set

[{'contract_id': 'SPARKLINGSPRINGWATERHOLDINGSLTD_07_03_2002-EX-10.13-SOFTWARE LICENSE AND MAINTENANCE AGREEMENT',
  'clause_category': 'Cap On Liability',
  'clause_text': 'In no event shall either party be liable to the other for the recovery of any special, indirect or consequential damages even if the defendant party had been advised of the possibility of such damages including but not limited to lost profits, lost revenues, failure to realize expected savings, loss of data and loss of use.',
  'tier': 2,
  'vague_terms': ['including but not limited to', 'limited'],
  'is_lethal': False,
  'has_context_risk': True,
  'reasoning_sentence': 'suit against its insurer,\nthe insurer is liable for: (1) the insured’s reasonable attorney’s fees in vindicating its claim; (2) the\ninsured’s damages for net economic loss caused by the delay in settlement, and damages for\naggravation and inconvenience.” Syl.'},
 {'contract_id': 'SPARKLINGSPRINGWATERHOLDINGSLTD_07_03_2002-EX-10.13-SOFTWARE LIC

Use a Cross-Encoder model to check the similarities in the extracted data

In [38]:
clause_texts = [data_point["clause_text"] for data_point in sample_triplet_set]
reasoning_texts = [data_point["reasoning_sentence"] for data_point in sample_triplet_set]

In [42]:
from sentence_transformers import CrossEncoder

ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

pairs = list(zip(clause_texts, reasoning_texts))

ce_scores = ce_model.predict(pairs)

for i, score in enumerate(ce_scores):
    print(f"Triplet {i+1} Cross-Encoder Score: {score:.4f}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Triplet 1 Cross-Encoder Score: -5.0773
Triplet 2 Cross-Encoder Score: -2.2213
Triplet 3 Cross-Encoder Score: -6.6665
Triplet 4 Cross-Encoder Score: -0.9792
Triplet 5 Cross-Encoder Score: -4.5738
Triplet 6 Cross-Encoder Score: -0.3349
Triplet 7 Cross-Encoder Score: -4.6939
Triplet 8 Cross-Encoder Score: -4.4263
Triplet 9 Cross-Encoder Score: -0.1464
Triplet 10 Cross-Encoder Score: 0.0761
Triplet 11 Cross-Encoder Score: -2.1078
Triplet 12 Cross-Encoder Score: -7.1219
